# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library, focusing on referencing all data entities by their `@id`.

### Dataset Source
The dataset describes protein abundance and modifications from human extracellular vesicles, with metadata and schema accessible via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets and each field's (column's) `@id`.

In [ ]:
# List record sets and their fields by `@id`
print("Available Record Sets (by @id):")
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"Record Set ID: {rs['@id']}")
        print(f"  Name: {rs.get('name', '')}")
        print("  Fields (by @id and name):")
        for field in rs.get('fields', []):
            print(f"    - {field.get('@id', '')} : {field.get('name', '')}")
        print()
# For notebook reference: Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []

## 3. Data Extraction
Load data from a specific record set into a DataFrame. All entities are referenced by their `@id` as shown above.

In [ ]:
# Extract data from each record set found
dataframes = {}

for record_set_id in record_set_ids:
    # Each record set may represent a table in the dataset
    print(f"Loading records from Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set columns (@id): {df.columns.tolist()}")
    print(f"Preview of records from {record_set_id}:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Process records using field `@id`. This includes filtering on a numeric field, normalization, and grouping.

We'll demonstrate with the first table (if available) and select a numeric field.

In [ ]:
if record_set_ids:
    # For demonstration, pick the first record set
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"Using Record Set: {main_record_set_id}")
    
    # Try to find numeric fields (float or integer) by checking types in metadata
    main_record_set_meta = next(rs for rs in metadata.record_sets if rs['@id'] == main_record_set_id)
    numeric_field_ids = [
        field['@id'] for field in main_record_set_meta.get('fields', [])
        if field.get('dataType', '').lower() in {'float', 'integer', 'number'}
    ]
    group_field_id = None
    # Optionally pick a group field (first text/categorical)
    for field in main_record_set_meta.get('fields', []):
        if field.get('dataType', '').lower() in {'string', 'text'}:
            group_field_id = field['@id']
            break
    
    # Proceed only if there's at least one numeric field
    if numeric_field_ids:
        numeric_field_id = numeric_field_ids[0]
        print(f"Numeric field selected (@id): {numeric_field_id}")
        # Convert column to numeric, if possible
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Example: filter values
        threshold = df[numeric_field_id].median() if df[numeric_field_id].notna().sum() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() + 1e-9)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by a categorical/text field if available
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped by {group_field_id}, showing means:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets found to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

We create a histogram and boxplot of the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_ids:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We have explored the FAIR^2 dataset, loading both metadata and records using `mlcroissant` and referencing all schema entities by their `@id`. We reviewed record sets and fields, extracted tabular data, performed filtering, normalization, and simple grouping, and visualized numeric distributions to highlight potential directions for advanced analysis.